# 05 · Timing & Profiling Your Code

**Module 12 – Standard Library Essentials** · Estimated time: 35 min

> Expands 📗 *Timing your code – timeit* with profilers that show **where** time is spent, including the 🆕 Python 3.15 `profiling` package and its sampling profiler, **Tachyon**.

### 🎯 What you will learn
- Measuring elapsed time with `time.perf_counter()`
- Benchmarking small snippets with `timeit` and Jupyter's `%timeit` / `%%timeit`
- Finding bottlenecks with a **profiler**: `cProfile` → 🆕 `profiling.tracing`
- 🆕 `profiling.sampling` (Tachyon, 3.15): low-overhead sampling profiler with flame graphs
- Measuring memory with `tracemalloc`

## 1. Example functions (📗)

Two ways to build the list `['0', '1', ..., 'n-1']`:

In [1]:
def func_one(n):
    """Given a number n, return a list of string digits from 0 to n."""
    return [str(num) for num in range(n)]

def func_two(n):
    """Same result using map()."""
    return list(map(str, range(n)))

print(func_one(10) == func_two(10))

True


## 2. Timing start and stop (📗)

In [2]:
import time

start_time = time.perf_counter()        # ✅ perf_counter: high-resolution, monotonic (📗 uses time.time())
result = func_one(1_000_000)
elapsed = time.perf_counter() - start_time
print(f"func_one: {elapsed:.4f}s")

start_time = time.perf_counter()
result = func_two(1_000_000)
print(f"func_two: {time.perf_counter() - start_time:.4f}s")

func_one: 0.0479s
func_two: 0.0382s


> ⚠️ `time.time()` (used in the bootcamp) measures wall-clock time and can jump if the system clock changes. For durations, use `time.perf_counter()`.

A single measurement is noisy (other programs, caches...). To compare small snippets reliably, run them **many times**: that's `timeit`.

## 3. The `timeit` module (📗)

In [3]:
import timeit

stmt = "func_one(100)"
setup = """
def func_one(n):
    return [str(num) for num in range(n)]
"""
print(timeit.timeit(stmt, setup, number=100_000))

stmt2 = "func_two(100)"
setup2 = """
def func_two(n):
    return list(map(str, range(n)))
"""
print(timeit.timeit(stmt2, setup2, number=100_000))

0.31447112499154173


0.30645737500162795


Easier: pass a **callable** or use `globals=` so you don't need setup strings:

In [4]:
print(timeit.timeit(lambda: func_one(100), number=100_000))
print(min(timeit.repeat("func_two(100)", globals=globals(), number=10_000, repeat=5)))

0.3144589159928728
0.030634208989795297


From the terminal: `python -m timeit "sum(range(1000))"`

## 4. Jupyter magic: `%timeit` and `%%timeit` (📗)

In [5]:
%timeit func_one(100)

3.15 μs ± 13.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [6]:
%timeit func_two(100)

3.13 μs ± 12.6 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [7]:
%%timeit
total = 0
for i in range(1000):
    total += i

8.37 μs ± 25.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


`%timeit` automatically picks the number of loops and reports mean ± standard deviation. (Magics only work in IPython/Jupyter.)

## 5. Profiling: *where* is the time spent?

Timing tells you *how long*; a **profiler** tells you *which functions* are responsible.

In [8]:
%%writefile slow_app.py
import time

def load_data():
    return [str(i) for i in range(200_000)]

def parse(rows):
    return [int(r) for r in rows]

def slow_report(nums):
    total = 0
    for n in nums:
        total += sum(range(n % 50))      # wasteful work
    return total

def main():
    rows = load_data()
    nums = parse(rows)
    time.sleep(0.05)
    print("report:", slow_report(nums))

if __name__ == "__main__":
    main()

Writing slow_app.py


### Deterministic (tracing) profiler: `cProfile` → 🆕 `profiling.tracing`

> 🆕 **Python 3.15:** the profilers were reorganised into a new `profiling` package. `cProfile` lives on as **`profiling.tracing`** (`cProfile` still works as an alias). The old pure-Python `profile` module is deprecated (removal planned for 3.17).

In [9]:
import subprocess, sys
out = subprocess.run([sys.executable, "-m", "profiling.tracing", "-s", "cumulative", "slow_app.py"],
                     capture_output=True, text=True).stdout
print("\n".join(out.splitlines()[:16]))

report: 78400000
         200009 function calls in 0.123 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.123    0.123 {built-in method builtins.exec}
        1    0.002    0.002    0.123    0.123 slow_app.py:1(<module>)
        1    0.000    0.000    0.121    0.121 slow_app.py:15(main)
        1    0.055    0.055    0.055    0.055 {built-in method time.sleep}
        1    0.022    0.022    0.043    0.043 slow_app.py:9(slow_report)
   200000    0.021    0.000    0.021    0.000 {built-in method builtins.sum}
        1    0.012    0.012    0.012    0.012 slow_app.py:3(load_data)
        1    0.011    0.011    0.011    0.011 slow_app.py:6(parse)
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
        1    0.000    0.000    0.000    0.000 {built-in method builtins.print}


Reading the table:
- **ncalls**: how many times the function was called
- **tottime**: time spent *in the function itself*
- **cumtime**: time *including* everything it called. Sort by this to find hot paths

Here `slow_report` dominates; that's where to optimise.

From Python code:

In [10]:
import cProfile, pstats, io, importlib
import slow_app

profiler = cProfile.Profile()
profiler.enable()
slow_app.main()
profiler.disable()
s = io.StringIO()
pstats.Stats(profiler, stream=s).sort_stats("tottime").print_stats(4)
print(s.getvalue()[:900])

report: 78400000
         200188 function calls (200187 primitive calls) in 0.144 seconds

   Ordered by: internal time
   List reduced from 69 to 4 due to restriction <4>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.041    0.021    0.123    0.061 {built-in method builtins.exec}
        1    0.033    0.033    0.033    0.033 {built-in method time.sleep}
        1    0.022    0.022    0.043    0.043 /private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmphwz7hbi2/slow_app.py:9(slow_report)
   200000    0.021    0.000    0.021    0.000 {built-in method builtins.sum}





### 🆕 Sampling profiler: `profiling.sampling` ("Tachyon", Python 3.15)

A **tracing** profiler records every function call, which adds overhead and distorts timings. A **sampling** profiler instead takes snapshots of the call stack thousands of times per second: very low overhead, safe for production, and it can even **attach to an already-running process**.

In [ ]:
!python -m profiling.sampling run slow_app.py                                   # profile a script
!python -m profiling.sampling run --flamegraph -o profile.html slow_app.py      # interactive flame graph
!python -m profiling.sampling run --live slow_app.py                            # live, top-like view
!python -m profiling.sampling attach 12345                                      # attach to a running PID

> 🔒 On **macOS** it needs elevated permissions (`sudo -E python -m profiling.sampling ...`); on Linux, ptrace permissions; on Windows, admin rights. The profiler's Python version must match the target's. That's why this cell isn't pre-run here.

Modes: `--mode wall` (default, wall-clock), `cpu`, `gil`, `exception`.

## 6. Measuring memory: `tracemalloc`

In [11]:
import tracemalloc

tracemalloc.start()
big_list = [str(i) for i in range(300_000)]
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"current: {current / 1e6:.1f} MB, peak: {peak / 1e6:.1f} MB")

tracemalloc.start()
total = sum(len(str(i)) for i in range(300_000))       # generator: no big list
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"generator version peak: {peak / 1e6:.3f} MB")

current: 16.6 MB, peak: 16.6 MB


generator version peak: 0.031 MB


## 7. Optimisation rules of thumb
1. **Make it work, make it right, then make it fast.** Don't optimise without measuring.
2. Profile first; optimise the **hot spots** only.
3. Algorithm and data-structure choices matter most (e.g. a `set` lookup vs a `list` scan).
4. Use built-ins (`sum`, `sorted`, `map`, comprehensions); they run in C.
5. Cache repeated work (`functools.cache`).
6. For heavy numeric work, use NumPy; for CPU-bound parallelism see Module 13.

## 📝 Summary
- `time.perf_counter()` for one-off durations.
- `timeit` / `%timeit` for reliable micro-benchmarks.
- `cProfile` = 🆕 `profiling.tracing` for function-level breakdowns; sort by `cumulative`.
- 🆕 `profiling.sampling` (Tachyon) for low-overhead profiling, flame graphs and attaching to live processes.
- `tracemalloc` for memory.

## 🏋️ Practice
1. Compare `"".join(list_of_strings)` vs `+=` concatenation for 10,000 strings with `timeit`.
2. Compare `x in list` vs `x in set` for 100,000 elements.
3. Profile a recursive Fibonacci (`fib(25)`) with `cProfile` and find the call count, then add `@functools.cache` and profile again.

<details><summary>Show solutions</summary>

```python
import timeit, cProfile, functools

words = ["word"] * 10_000
def concat():
    s = ""
    for w in words:
        s += w
    return s
print(timeit.timeit(lambda: "".join(words), number=200), timeit.timeit(concat, number=200))

data = list(range(100_000)); s = set(data)
print(timeit.timeit(lambda: 99_999 in data, number=1000), timeit.timeit(lambda: 99_999 in s, number=1000))

def fib(n): return n if n < 2 else fib(n - 1) + fib(n - 2)
cProfile.run("fib(25)")                   # ~250,000 calls
fib = functools.cache(fib)
cProfile.run("fib(25)")                   # ~26 calls
```
</details>